<a href="https://colab.research.google.com/github/StAandrew/listing-parser/blob/main/Listing_Parser_Fine_Tune_Unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Listing description fine-tuning

# Installation

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install "transformers>=4.57,<4.60"
!pip install --no-deps trl==0.22.2
!pip install --quiet wandb weave
!pip install --quiet git+https://github.com/StAandrew/listing-parser.git

# Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192 # Choose any! We auto support RoPE Scaling internally!
# 8192 because: our system prompt is ~5.5k tokens (closed-vocab
# reinforcement + 3 few-shot examples from examples.json), plus the
# user message (typical listing 500-2000 tokens) plus the expected
# assistant JSON (400-1200 tokens). 4k would silently truncate long
# rows — the label loss would still compute but on a clipped
# sequence, quietly biasing the fine-tune toward short listings.

dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
# Note: some model names are outdated
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",      # New Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",           # Llama-3 15 trillion tokens model 2x faster!
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",        # Phi-3 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",             # Gemma 2.2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    # r=32 (vs tutorial default 16): structured-extraction tasks on
    # closed vocabularies benefit from more adapter capacity than
    # chat-style fine-tunes. Bump to 64 if the loss plateaus above
    # ~0.4; drop to 16 if you're tight on VRAM.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # alpha == r is the stable starting point
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Trainable params: 83,886,080 / 4,624,486,400 (1.81%)


# Data prep

In [5]:
from datasets import load_dataset

dataset = load_dataset("standrey/listing-descriptions", split = "train")
print(dataset.column_names)

README.md: 0.00B [00:00, ?B/s]

data/batch_00000.parquet:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1704 [00:00<?, ? examples/s]

['description', 'listing_type', 'output']


# Prepare the dataset

In [6]:
import urllib.request, os

# prompt.md and examples.json aren't bundled with the pip-installed
# package — they're at the repo root, not inside src/listing_parser/.
# Download them so `build_system_prompt()` finds them via its
# Path.cwd() fallback.
for fname in ("prompt.md", "examples.json"):
    if not os.path.exists(fname):
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/StAandrew/listing-parser/main/src/listing_parser/_assets/{fname}",
            fname,
        )
        print(f"downloaded {fname}")
    else:
        print(f"{fname} already present")

downloaded prompt.md
downloaded examples.json


In [7]:
from datasets import load_dataset
from listing_parser.prompting import build_system_prompt, build_user_message

# Pull the same 1,704 rows we pushed to HF via scripts/push_labels_to_hf.py.
# The dataset has three columns: description, listing_type, output.
# `output` is the teacher's fenced JSON (e.g. "```json\n{...}\n```").
dataset = load_dataset("standrey/listing-descriptions", split="train")
print("Columns:", dataset.column_names)
print("Rows:   ", len(dataset))
print("Per-type histogram:",
      {t: sum(1 for r in dataset if r["listing_type"] == t)
       for t in ("Rent", "Sale", "Room")})

# Build the system prompt ONCE — it's ~5.5k deterministic tokens
# rendered from prompt.md + schema vocab + examples.json. Every row
# shares it, which is why Bedrock's prompt caching makes the teacher
# runs effectively free after warm-up.
SYSTEM_PROMPT = build_system_prompt()
print(f"\nSystem prompt length (chars): {len(SYSTEM_PROMPT):,}")

def to_messages(row):
    """Convert one HF row into the {messages: [...]} format SFTTrainer expects.

    We keep the assistant response fenced exactly the way the teacher
    emitted it, because the runner's parse_output_json strips fences at
    inference time. Training the model to emit fences means the fine-
    tune's raw output matches production inference verbatim.
    """
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": build_user_message(
                row["description"], row["listing_type"])},
            {"role": "assistant", "content": row["output"]},
        ]
    }

formatted = dataset.map(to_messages, remove_columns=dataset.column_names)

# Apply the Llama 3.1 chat template to turn {messages: [...]} into a
# single `text` string with the correct <|begin_of_text|>/<|eot_id|>
# tokens. Unsloth handles Llama-3.1's template; no tokenizer hack needed.
def apply_template(row):
    return {
        "text": tokenizer.apply_chat_template(
            row["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

formatted = formatted.map(apply_template, remove_columns=["messages"])

# Token-length diagnostic — if the p99 is close to max_seq_length we
# are losing training signal on the longest rows. 8192 should be
# comfortable but let's verify.
import numpy as np
lens = [len(tokenizer(r["text"]).input_ids) for r in formatted.select(range(min(200, len(formatted))))]
print(f"\nSample (n={len(lens)}) token-length: "
      f"median={int(np.median(lens))}, p95={int(np.percentile(lens,95))}, "
      f"max={max(lens)}")

# Peek at one fully-rendered example to confirm the template worked.
print("\n----- formatted[0] first 800 chars -----")
print(formatted[0]["text"][:800])
print("... (truncated)")

# 95/5 split. The "real" eval is the 60-row gold set scored with the
# listing-parser scorer — this small holdout only gives us a training-
# time loss curve to watch for overfitting.
split = formatted.train_test_split(test_size=0.05, seed=3407)
train_ds, eval_ds = split["train"], split["test"]
print(f"\nTrain: {len(train_ds)} rows, eval: {len(eval_ds)} rows")

Columns: ['description', 'listing_type', 'output']
Rows:    1704
Per-type histogram: {'Rent': 983, 'Sale': 205, 'Room': 516}

System prompt length (chars): 19,922


Map:   0%|          | 0/1704 [00:00<?, ? examples/s]

Map:   0%|          | 0/1704 [00:00<?, ? examples/s]


Sample (n=200) token-length: median=5697, p95=6342, max=7054

----- formatted[0] first 800 chars -----
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a UK property listing description parser. You receive one listing
description (free text, often from Rightmove, Zoopla, or SpareRoom) and
return a single JSON object that extracts structured facts about the
property. The JSON MUST conform to the schema below.

General rules:
  - Return JSON only. No prose, no markdown, no code fences.
  - Extract only facts stated or clearly implied by the description. Never
    guess, never invent. When a field is not mentioned, OMIT it from the
    output (do not output null, do not output "" — leave the key out).
  - Booleans mean "the description states this feature is present / true".
    Do NOT output `false` for things 
... (truncated)

Train: 1618 rows, eval: 86 rows


# Train

First, initiate Wandb

In [8]:
import os

# Pull the key from Colab secrets FIRST, before importing wandb.
# Belt-and-braces: also accept a plain env var so the cell works
# outside Colab.
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except (ImportError, Exception):
    pass  # not in Colab, or secret not set

if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError(
        "WANDB_API_KEY not set. Add it via Colab's 🔑 sidebar "
        "(name: WANDB_API_KEY) or export it as an env var."
    )

import wandb
import weave

# host=None uses the default public cloud. force=True skips the
# "already logged in / not logged in" heuristic that's been flaky
# on Colab runtimes — we just authenticate with the env var.
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

RUN_NAME = "llama-3.1-8b-ft-v1-quick"
wandb.init(
    project = "listing-parser",
    name    = RUN_NAME,
    group   = "llama-3.1-8b-ft-v1",
    job_type= "train",
    config  = {
        "base_model":    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        "dataset":       "standrey/listing-descriptions",
        "n_train_rows":  len(train_ds),
        "n_eval_rows":   len(eval_ds),
        "max_seq_length": max_seq_length,
        "lora_r":        32,
        "lora_alpha":    32,
        "learning_rate": 2e-4,
        "batch_size":    1,
        "grad_accum":    8,
        "phase":         "quick",
    },
    reinit = True,
)
print(f"W&B run: {wandb.run.url}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: staandrew to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Initializing weave.
weave: wandb version 0.26.1 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: Logged in as Weights & Biases user: staandrew.
weave: View Weave data at https://wandb.ai/staandrew/listing-parser/weave


W&B run: https://wandb.ai/staandrew/listing-parser/runs/m1byv4h1


Actual training

In [25]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported
import wandb

# Phase 1 (quick smoke) vs Phase 2 (full run). You'll execute this
# cell twice: first with PHASE="quick", eyeball the loss curve, then
# with PHASE="full". Setting resume_from_checkpoint=True on the second
# pass continues from wherever phase 1 left off — no wasted compute.
PHASE = "full"  # change to "full" for the second pass

if PHASE == "quick":
    training_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,         # effective batch = 8
        warmup_steps = 20,
        max_steps = 300,                         # ~25 min on T4
        learning_rate = 2e-4,
        logging_steps = 10,
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 3,
        eval_strategy = "steps",
        eval_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
    )
elif PHASE == "full":
    training_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 40,
        num_train_epochs = 2,                    # ~800 steps @ 1.6k rows
        learning_rate = 2e-4,
        logging_steps = 10,
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 3,
        eval_strategy = "steps",
        eval_steps = 100,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
    )
else:
    raise ValueError(f"unknown PHASE={PHASE!r}")

print("active wandb run:", wandb.run.id if wandb.run else "NONE")
print("active wandb url:", wandb.run.url if wandb.run else "NONE")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,                # packing risks mixing unrelated
                                    # (description, output) pairs across
                                    # a sequence boundary, which is bad
                                    # for structured-extraction losses.
    args = training_args,
)

# Resume from the quick-phase checkpoint if it exists.
import os
resume = os.path.isdir("outputs") and any(
    d.startswith("checkpoint-") for d in os.listdir("outputs")
)
print(f"PHASE={PHASE}, resume={resume}")

trainer_stats = trainer.train(resume_from_checkpoint=False)

active wandb run: m1byv4h1
active wandb url: https://wandb.ai/staandrew/listing-parser/runs/m1byv4h1
PHASE=full, resume=False


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,618 | Num Epochs = 2 | Total steps = 406
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss,Validation Loss
100,0.001400,0.001360
200,0.001400,0.001363
300,0.001400,0.001359
400,0.001400,0.001359


In [23]:
!rm -rf outputs/

# Save merged bf16 weights

In [ ]:
# Merge LoRA adapters into the base model, save at bf16. This is the
# artefact Bedrock Custom Model Import ingests — safetensors + config
# + tokenizer files in the HF layout. ~16GB on disk; Colab's /content
# has room.
MERGED_DIR = "merged_bf16"

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",   # bf16 if supported, else fp16
)

# Also push to HF. Having the weights at a stable URL means Bedrock
# CMI can pull directly from HF (or you can upload the tarball — HF
# is easier). Use a versioned repo name so ft-v1, ft-v2, etc. coexist.
from huggingface_hub import login
import os
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    from google.colab import userdata  # secret manager in Colab
    login(token=userdata.get("HF_TOKEN"))

model.push_to_hub_merged(
    "standrey/listing-parser-llama31-8b-ft-v1",
    tokenizer,
    save_method = "merged_16bit",
    private = False,   # flip to False once you're happy with it
)

print(f"\nMerged weights in {MERGED_DIR}")
print("Pushed to: https://huggingface.co/standrey/listing-parser-llama31-8b-ft-v1")

# Score against the frozen gold set

In [ ]:
import json, re, urllib.request
from pathlib import Path

# Download the frozen gold set from the repo. Pinning to main is OK
# — test_set.jsonl is immutable by policy (CLAUDE.md priority #1).
GOLD_URL = "https://raw.githubusercontent.com/StAandrew/listing-parser/main/benchmarks/test_set.jsonl"
gold_text = urllib.request.urlopen(GOLD_URL).read().decode("utf-8")
gold_rows = [json.loads(line) for line in gold_text.splitlines() if line.strip()]
print(f"Loaded {len(gold_rows)} gold rows")

# Flip into 2x-faster inference mode. Mandatory for Unsloth-wrapped
# models — without this, the model runs in train mode and inference
# is 2-3x slower.
FastLanguageModel.for_inference(model)

# Reuse the repo's parser so the notebook's predictions.jsonl uses
# the exact same fence-stripping rules the scorer expects.
from listing_parser.cleaning import parse_output_json

def predict_one(description: str, listing_type: str) -> tuple[dict | None, str]:
    """Mirrors BedrockLlamaRunner.predict() semantics: greedy decode,
    no schema retry, single parse attempt. We want the honest first-
    pass signal, not a result inflated by retries."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": build_user_message(description, listing_type)},
    ]
    # add_generation_prompt=True appends the <|start_header_id|>assistant<...>
    # tokens so the model knows it's its turn to speak.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids = inputs,
            max_new_tokens = 1536,   # max expected output ~1200 tokens
            do_sample = False,       # greedy — reproducible, matches
                                     # BedrockLlamaRunner temperature=0
            use_cache = True,
            pad_token_id = tokenizer.eos_token_id,
        )
    # Strip the prompt tokens; we only want what the model generated.
    raw = tokenizer.decode(out[0, inputs.shape[1]:], skip_special_tokens=True)
    pred = parse_output_json(raw)
    return pred, raw

# Run all 60 rows. T4 @ 8B bf16 gives ~30-60s/row → ~45min total.
# A100 is ~2-3x faster. Progress prints because Colab's buffering
# can make long cells look dead.
import time
out_path = Path("predictions.jsonl")
with out_path.open("w", encoding="utf-8") as f:
    t0 = time.monotonic()
    for i, row in enumerate(gold_rows):
        pred, raw = predict_one(row["description"], row["listing_type"])
        f.write(json.dumps({
            "row_index": row["row_index"],
            "pred": pred,
            "raw": raw,
        }, ensure_ascii=False) + "\n")
        f.flush()
        if (i + 1) % 5 == 0:
            rate = (i + 1) / (time.monotonic() - t0)
            eta = (len(gold_rows) - i - 1) / rate
            print(f"  {i+1}/{len(gold_rows)}  rate={rate:.2f}/s  eta={eta/60:.1f}min")

print(f"\nWrote {out_path} ({out_path.stat().st_size:,} bytes)")

# Download predictions for local scoring

In [ ]:
from google.colab import files
files.download("predictions.jsonl")

Run locally

```
# lp-benchmark score \
    --gold benchmarks/test_set.jsonl \
    --predictions benchmarks/runs/llama-3.1-8b-ft-v1-colab/predictions.jsonl \
    --out-dir benchmarks/runs/llama-3.1-8b-ft-v1-colab \
    --name "llama-3.1-8b ft-v1 (Colab)"
```



# Export to GGUF for Ollama

In [ ]:
# Gate this — GGUF conversion is slow (5-15min) and is not what
# Bedrock needs. Set to True only if you want a laptop Ollama copy.
EXPORT_GGUF = False

if EXPORT_GGUF:
    model.save_pretrained_gguf(
        "gguf_q4_k_m",
        tokenizer,
        quantization_method = "q4_k_m",  # same quant level the user's
                                         # local Ollama baseline uses
    )
    # Optional HF push of the GGUF — handy for `ollama pull hf.co/...`:
    model.push_to_hub_gguf(
        "standrey/listing-parser-llama31-8b-ft-v1-gguf",
        tokenizer,
        quantization_method = "q4_k_m",
        private = True,
    )
else:
    print("GGUF export skipped. Set EXPORT_GGUF=True to enable.")